In [1]:
# Google Colab Notebook: CPU vs GPU Execution Comparison
# Assignment: Deep Learning Model Training Performance Analysis
!pip install -q transformers datasets tensorflow tensorflow_datasets
!pip install numpy==1.26.4

In [2]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
print("Is GPU being used by default:", tf.test.is_built_with_cuda())

TensorFlow version: 2.18.0
Num GPUs Available: 1
Is GPU being used by default: True


1. Las librerias que se utilizarán

In [3]:
import tensorflow as tf
import pandas as pd
from transformers import BertTokenizer, TFBertForSequenceClassification, TFDistilBertForSequenceClassification
from transformers import DataCollatorWithPadding, create_optimizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
import zipfile
import pandas as pd
import time
import numpy as np
from sklearn.metrics import accuracy_score

2. Iniciamos con el código, con sus respectivos pasos

In [11]:
#dataset = load_dataset("google/jigsaw_toxicity_pred")
#Este error ocurre porque el conjunto de datos jigsaw_toxicity_pred
#no se puede descargar automáticamente utilizando la función load_dataset
#de la biblioteca datasets. Requiere que descargues los datos manualmente desde Kaggle
#y luego especifiques la ruta donde guardaste los archivos descargados cuando llames a load_dataset.


In [4]:
from google.colab import files

# Esto abrirá una ventana para seleccionar archivos desde tu computadora
uploaded = files.upload()

Saving test.csv to test (1).csv
Saving test_labels.csv to test_labels (1).csv
Saving train.csv to train (1).csv


In [15]:
#Cargar la data desde Kaggle TRAIN
dt_train = pd.read_csv("train.csv")[['comment_text', 'toxic']].dropna()
dt_train = dt_train.sample(10, random_state=42)
dt_train['label'] = (dt_train['toxic'] >= 0.5).astype(int)

#Cargar la data desde Kaggle TEST y TEST_LABEL
dt_test = pd.read_csv("test.csv")[['comment_text', 'id']].dropna()
dt_test_label = pd.read_csv("test_labels.csv")[['id', 'toxic']].dropna()

# Fusionar primero para asegurar que todas las filas tengan etiquetas
dt_test = dt_test.merge(dt_test_label, on='id', how='inner')
dt_test['label'] = (dt_test['toxic'] >= 0.5).astype(int)
dt_test = dt_test.sample(5, random_state=42)  # Muestrear después de fusionar


# ------------------------
# 2. Tokenización
# -----------------------
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
def tokenizar(examples):
    return tokenizer(examples["comment_text"],
                     padding="max_length",
                     truncation=True,
                     max_length=512)

from datasets import Dataset, Features, Value

# Convertir DataFrames de pandas a objetos Dataset
dataset_train = Dataset.from_pandas(dt_train)
dataset_test = Dataset.from_pandas(dt_test)

tokenized_train = dataset_train.map(tokenizar, batched=True)
tokenized_test = dataset_test.map(tokenizar, batched=True)


# Preparar datasets para TensorFlow
tokenized_train = tokenized_train.remove_columns(['comment_text', '__index_level_0__', 'toxic'])
tokenized_train.set_format("tf", columns=["input_ids", "attention_mask", "label"])

tokenized_test = tokenized_test.remove_columns(['comment_text', '__index_level_0__', 'toxic', 'id'])
tokenized_test.set_format("tf", columns=["input_ids", "attention_mask", "label"])

#Colocar los datos como tensores
tf_train_dataset = tokenized_train.to_tf_dataset(
    columns=["attention_mask", "input_ids"],
    label_cols=["label"],
    shuffle=True,
    batch_size=16,
)

tf_test_dataset = tokenized_test.to_tf_dataset(
    columns=["attention_mask", "input_ids"],
    label_cols=["label"],
    shuffle=False,
    batch_size=16,
)



# ---------------------------------
# 3. Modelo de clasificación binaria
# ---------------------------------

def train_and_predict(device_name, tf_train_dataset, tf_test_dataset):
    print(f"\n{'='*60}\nIniciando entrenamiento en {device_name}\n{'='*60}")

    with tf.device(device_name):
        model = TFDistilBertForSequenceClassification.from_pretrained(
            "distilbert-base-uncased",
            num_labels=2
        )

        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),
            loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
            metrics=['accuracy']
        )
        # Entrenamiento
        start_time = time.time()
        history = model.fit(
            tf_train_dataset,
            validation_data=tf_test_dataset,
            epochs=2
        )
        train_time = time.time() - start_time

        # Predicción
        start_pred = time.time()
        predictions = model.predict(tf_test_dataset)
        pred_time = time.time() - start_pred

        # Procesar resultados
        logits = predictions.logits
        probabilities = tf.nn.softmax(logits, axis=-1).numpy()
        toxict_probs = probabilities[:, 1]
        predicted_labels = (toxict_probs >= 0.5).astype(int)

        # Obtener etiquetas reales
        true_labels = np.concatenate([y.numpy() for x, y in tf_test_dataset])

        # Calcular el tiempo total
        total_time = train_time + pred_time

        return {
            'model': model,
            'train_time': train_time,
            'pred_time': pred_time,
            'probabilities': toxict_probs,
            'predicted_labels': predicted_labels,
            'true_labels': true_labels,
            'total_time': total_time,
            'history': history.history
        }

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/datasets/arrow_dataset.py:400: FutureWarning: The output of `to_tf_dataset` will change when a passing single element list for `labels` or `columns` in the next datasets version. To return a tuple structure rather than dict, pass a single string.
Old behaviour: columns=['a'], labels=['labels'] -> (tf.Tensor, tf.Tensor)  
             : columns='a', labels='labels' -> (tf.Tensor, tf.Tensor)  
New behaviour: columns=['a'],labels=['labels'] -> ({'a': tf.Tensor}, {'labels': tf.Tensor})  
             : columns='a', labels='labels' -> (tf.Tensor, tf.Tensor) 
  warnings.warn(


In [16]:

# ---------------------------------
# 4. Resultados del modelo
#----------------------------------
results = {}

if tf.config.list_physical_devices('GPU'):
    results['gpu'] = train_and_predict('/GPU:0', tf_train_dataset, tf_test_dataset)
else:
    print("GPU no disponible")

results['cpu'] = train_and_predict('/CPU:0', tf_train_dataset, tf_test_dataset)

# ---------------------------------
# 5. Comparación de CPU vs. GPU
#----------------------------------


print("\nCOMPARACIÓN FINAL:")
print(f"Tiempo CPU: {results['cpu']['total_time']:.2f}s | Tiempo GPU: {results.get('gpu', {}).get('total_time', 'N/A')}")
if 'gpu' in results:
    speedup = results['cpu']['total_time'] / results['gpu']['total_time']
    print(f"Speedup GPU: {speedup:.2f}x")


Iniciando entrenamiento en /GPU:0


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_transform.bias', 'vocab_projector.bias', 'vocab_layer_norm.weight', 'vocab_transform.weight', 'vocab_layer_norm.bias']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
You should 

Epoch 1/2
1/1 [==============================] - 23s 23s/step - loss: 0.6627 - accuracy: 0.8000 - val_loss: 0.5815 - val_accuracy: 1.0000
Epoch 2/2
1/1 [==============================] - 2s 2s/step

Iniciando entrenamiento en /CPU:0


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_transform.bias', 'vocab_projector.bias', 'vocab_layer_norm.weight', 'vocab_transform.weight', 'vocab_layer_norm.bias']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
You should 

Epoch 1/2


1/1 [==============================] - ETA: 0s - loss: 0.7333 - accuracy: 0.1000

1/1 [==============================] - 50s 50s/step - loss: 0.7333 - accuracy: 0.1000 - val_loss: 0.6638 - val_accuracy: 1.0000
Epoch 2/2
1/1 [==============================] - 28s 28s/step - loss: 0.6449 - accuracy: 0.9000 - val_loss: 0.5900 - val_accuracy: 1.0000


1/1 [==============================] - 5s 5s/step

COMPARACIÓN FINAL:
Tiempo CPU: 102.66s | Tiempo GPU: 26.16635489463806
Speedup GPU: 3.92x
